# 05 - Protocol Design and Splitter Validation

**Purpose:** Build and validate the P0-P6 splitters with assertions and a visual timeline plot per protocol.

**Inputs:** configs/protocols.yaml; feature matrices from notebook 04.

**Outputs:** Validated splitter behavior; protocol timeline figures under results/figures/.

**Runtime estimate:** 5-10 minutes.

## Load protocol configuration

In [1]:
import pandas as pd

from sdg_leakage_audit.protocols.registry import PROTOCOL_REGISTRY, get_protocol
from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_processed_dir, results_figures_dir

protocols_cfg = read_yaml(configs_dir() / "protocols.yaml")["protocols"]
datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]
figures_dir = results_figures_dir()

print(f"{len(protocols_cfg)} protocols configured: {list(protocols_cfg.keys())}")
print(f"{len(PROTOCOL_REGISTRY)} protocols registered in code: {list(PROTOCOL_REGISTRY.keys())}")
assert set(protocols_cfg.keys()) == set(PROTOCOL_REGISTRY.keys()), "Config/registry protocol id mismatch!"

# Use household_power_consumption's feature matrix as the reference series for splitter
# validation below -- any dataset would do, since the splitters operate on positions, not values.
reference_df = pd.read_parquet(data_processed_dir() / "household_power_consumption_features.parquet")
reference_horizon = datasets_cfg["household_power_consumption"]["forecast_horizons"][0]
print(f"Reference series: {len(reference_df):,} rows, horizon={reference_horizon}")


7 protocols configured: ['P0', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6']
7 protocols registered in code: ['P0', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6']


Reference series: 2,075,259 rows, horizon=1


## Validate P0 chronological split with purge gap

In [2]:
from sdg_leakage_audit.protocols.splitters import chronological_split_with_gap

train_idx, test_idx = chronological_split_with_gap(
    reference_df, train_frac=0.6, test_frac=0.2, horizon=reference_horizon
)

# Leakage guard assertions (must hold for every P0 split, on every dataset):
assert len(set(train_idx) & set(test_idx)) == 0, "train/test index overlap detected"
assert train_idx.max() < test_idx.min(), "train positions must all precede test positions"
assert test_idx.min() - train_idx.max() - 1 >= reference_horizon, "purge gap smaller than horizon"

print(
    f"P0 split OK: train=[0, {train_idx.max()}], "
    f"gap=[{train_idx.max() + 1}, {test_idx.min() - 1}] ({test_idx.min() - train_idx.max() - 1} positions), "
    f"test=[{test_idx.min()}, {test_idx.max()}]"
)


P0 split OK: train=[0, 1245154], gap=[1245155, 1245155] (1 positions), test=[1245156, 1660207]


## Validate P1 random split

In [3]:
from sdg_leakage_audit.protocols.splitters import random_split

p1_train_idx, p1_test_idx = random_split(reference_df, train_frac=0.6, test_frac=0.2, seed=0)

assert len(set(p1_train_idx) & set(p1_test_idx)) == 0, "train/test index overlap detected"

# The key contrast with P0: P1's test positions are NOT all after train positions --
# this is exactly the "breaks the arrow of time" property under audit.
is_chronological = p1_train_idx.max() < p1_test_idx.min()
n_test_before_train_max = int((p1_test_idx < p1_train_idx.max()).sum())
print(f"P1 split: train/test disjoint (OK), chronological order preserved = {is_chronological}")
print(
    f"{n_test_before_train_max} of {len(p1_test_idx)} test positions fall BEFORE the latest train "
    "position -- i.e. interleaved with train in time, confirming P1 breaks chronological order."
)


P1 split: train/test disjoint (OK), chronological order preserved = False
415052 of 415052 test positions fall BEFORE the latest train position -- i.e. interleaved with train in time, confirming P1 breaks chronological order.


## Validate P6 blocked vs shuffled CV

In [4]:
from sdg_leakage_audit.protocols.splitters import blocked_cv_splits, shuffled_kfold_splits

n_folds = protocols_cfg["P6"]["params"]["n_folds"]
blocked_folds = blocked_cv_splits(reference_df, n_folds=n_folds, horizon=reference_horizon, expanding=True)
shuffled_folds = shuffled_kfold_splits(reference_df, n_folds=n_folds, seed=0)

for fold_i, (train_idx, test_idx) in enumerate(blocked_folds):
    assert len(set(train_idx) & set(test_idx)) == 0
    assert train_idx.max() < test_idx.min(), f"blocked fold {fold_i}: train must precede test"

interleaved_fold_count = 0
for fold_i, (train_idx, test_idx) in enumerate(shuffled_folds):
    assert len(set(train_idx) & set(test_idx)) == 0
    if train_idx.max() > test_idx.min():  # some train position falls after some test position
        interleaved_fold_count += 1

print(f"Blocked/expanding CV: {len(blocked_folds)} folds, all respect chronological order + purge gap.")
print(
    f"Shuffled K-fold CV: {len(shuffled_folds)} folds, {interleaved_fold_count}/{len(shuffled_folds)} "
    "folds have train positions interleaved with/after test positions -- confirming P6 breaks "
    "chronological order within folds."
)


Blocked/expanding CV: 5 folds, all respect chronological order + purge gap.
Shuffled K-fold CV: 5 folds, 5/5 folds have train positions interleaved with/after test positions -- confirming P6 breaks chronological order within folds.


## Protocol timeline visualization

In [5]:
from sdg_leakage_audit.viz.plots import plot_protocol_timeline
from sdg_leakage_audit.viz.style import apply_plot_style

apply_plot_style()

ILLUSTRATIVE_N = 300  # small illustrative series length, independent of any real dataset's size

for protocol_id in PROTOCOL_REGISTRY:
    plot_protocol_timeline(protocol_id, n=ILLUSTRATIVE_N, save_path=figures_dir / f"05_timeline_{protocol_id}.png")

print(f"Saved {len(PROTOCOL_REGISTRY)} protocol timeline figures to results/figures/.")


2026-10-19 23:51:13 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P0.png


2026-10-19 23:51:14 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P1.png


2026-10-19 23:51:14 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P2.png


2026-10-19 23:51:14 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P3.png


2026-10-19 23:51:15 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P4.png


2026-10-19 23:51:15 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P5.png


2026-10-19 23:51:16 | INFO     | sdg_leakage_audit.viz.plots | Saved figure -> results\figures\05_timeline_P6.png


Saved 7 protocol timeline figures to results/figures/.


## Findings / what to carry forward

- All 7 protocols (P0-P6) are registered consistently between `configs/protocols.yaml` and
  `protocols/registry.py` (checked by assertion in this notebook).
- P0's chronological split with a purge gap >= horizon was validated on the real
  household-power feature matrix: no train/test overlap, train strictly precedes test, and the
  gap is at least `reference_horizon` positions -- this is the automated leakage guard the
  project requires, now exercised against real data rather than only synthetic test fixtures.
- P1 (random split) and P6 (shuffled K-fold) were both confirmed to break chronological order
  on the same real data: P1 has test positions interleaved with train positions in time; P6's
  shuffled folds mix train/test temporally, unlike blocked/expanding CV.
- Protocol timeline figures for all 7 protocols saved to `results/figures/05_timeline_P*.png` --
  P0/P2/P3/P4/P5 share the same underlying chronological split (confirmed in
  `configs/protocols.yaml`'s params; only P3 drops the purge gap), so they're rendered as the
  same train/gap/test bar with a protocol-specific annotation showing exactly where that
  protocol's leak occurs (fit span for P2, overlap region for P3, feature-reads-ahead arrow for
  P4, test-set-scoring arrow for P5) -- P1 and P6 get dedicated visualizations since their split
  structure genuinely differs.
- Carry forward to notebook 06: the validated P0 split (and `reference_horizon` convention) is
  exactly what the P0 baseline models will use; notebook 07 will swap in the P1-P6 variants
  validated here to measure their inflation effect.
